# Feature Engineering Parque Solar Girasol

In [2]:
#### 1. IMPORTS Y CONFIGURACIÓN
import os
import pandas as pd
import numpy as np
import math
import sys # Import sys module
import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", category=PerformanceWarning)
import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.filterwarnings("ignore", category=InterpolationWarning)


# Add the 'src' directory to Python's path so it can find your new module.
# The '..' means go up one directory from 'notebooks' to the repository root.
sys.path.append(os.path.abspath('../src'))

# Now, import your custom class and helper functions from the new 'feature_engineer' module
from feature_engineer import SolarFeatureEngineer, identify_non_stationary, compute_best_lags, add_temporal_features

# Keep other necessary imports that are used elsewhere in your notebook
# (e.g., for data loading, model training, evaluation, etc.)
from statsmodels.tsa.stattools import adfuller, kpss
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

# ... The rest of your notebook code that loads data, configures sfe, fits, transforms, and saves
# Example:
# input_path = "../data/interim/meteo_data_with_generation_clean/parque_solar_girasol_clean.parquet"
# df = pd.read_parquet(input_path)
# sfe = SolarFeatureEngineer(...)
# sfe.fit(df)
# joblib.dump(sfe, "path/to/models/solar_feature_engineer.joblib")

#### 1. CARGA DE DATOS LIMPIOS

In [4]:
input_path = "../data/interim/meteo_data_with_generation_clean/parque_solar_girasol_clean.parquet"
df = pd.read_parquet(input_path)

In [5]:
len(df)

32061

#### 2. CONFIGURACIÓN DE FEATURE ENGINEERING

In [6]:
log_cols = [
    'shortwave_radiation','diffuse_radiation',
    'global_tilted_irradiance','shortwave_radiation_instant',
    'diffuse_radiation_instant','global_tilted_irradiance_instant',
    'direct_radiation','direct_normal_irradiance',
    'wind_speed_10m','wind_gusts_10m',
    'vapour_pressure_deficit','sunshine_duration'
]

sfe = SolarFeatureEngineer(
    target='generation',
    max_lag=24,
    roll_windows=[3,6,24],
    log_transform_cols=log_cols
)

sfe.fit(df)

df_feat = sfe.transform(df)

sfe.get_feature_names_out()

['date',
 'generation',
 'cloud_cover',
 'cloud_cover_low',
 'cloud_cover_mid',
 'et0_fao_evapotranspiration',
 'sunshine_duration',
 'shortwave_radiation',
 'global_tilted_irradiance',
 'shortwave_radiation_instant',
 'global_tilted_irradiance_instant',
 'direct_radiation',
 'direct_normal_irradiance',
 'terrestrial_radiation',
 'direct_radiation_instant',
 'direct_normal_irradiance_instant',
 'terrestrial_radiation_instant',
 'temperature_2m_diff1',
 'dew_point_2m_diff1',
 'relative_humidity_2m_diff1',
 'apparent_temperature_diff1',
 'surface_pressure_diff1',
 'vapour_pressure_deficit_diff1',
 'wind_speed_10m_diff1',
 'wind_direction_10m_diff1',
 'wind_gusts_10m_diff1',
 'is_day_diff1',
 'wet_bulb_temperature_2m_diff1',
 'diffuse_radiation_diff1',
 'diffuse_radiation_instant_diff1',
 'pressure_msl_diff1',
 'temperature_2m_lag0',
 'dew_point_2m_lag4',
 'relative_humidity_2m_lag0',
 'apparent_temperature_lag0',
 'surface_pressure_lag3',
 'cloud_cover_lag6',
 'cloud_cover_low_lag15',
 '

#### 3. GENERACIÓN DEL DATASET DE FEATURES

In [7]:
df_feat = sfe.transform(df)

In [5]:
import os
import joblib

# 5.1 Asegúrate de que exista la carpeta de modelos
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

# 5.2 Guarda el transformer ya “fiteado”
fe_path = os.path.join(models_dir, "solar_feature_engineer.joblib")
joblib.dump(sfe, fe_path)

print(f"✅ Transformer guardado en: {fe_path}")

✅ Transformer guardado en: ../models\solar_feature_engineer.joblib


#### 4. GUARDADO DEL DATASET PROCESADO

In [6]:
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "parque_solar_girasol_model_ready.parquet")
df_feat.to_parquet(output_path)

print(f"Features generadas y guardadas en: {output_path}")

Features generadas y guardadas en: ../data/processed\parque_solar_girasol_model_ready.parquet
